# Part C · Indic Token Behavior Analysis
**Goal:** Deep-dive into how each tokenizer handles Tamil's agglutinative morphology — vocabulary coverage, subword fragmentation, memory pressure, and characters-per-token quality.

**Data source:** `../part_b_token_analysis/token_counts.csv` (written by Part B)

In [ ]:
# ── Cell 0 · Runtime check + global visual theme ──────────────────────────────
import sys, os, warnings
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import numpy as np
import pandas as pd
from IPython.display import display, HTML
warnings.filterwarnings("ignore")

print("Python :", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA   :", torch.cuda.is_available())
os.makedirs("plots", exist_ok=True)

PALETTE = ["#2E86AB", "#A23B72", "#F18F01", "#C73E1D", "#3B1F2B"]
MODEL_COLORS = {
    "IndicTrans2" : "#2E86AB",
    "NLLB-200"    : "#A23B72",
    "mT5"         : "#F18F01",
    "Helsinki"    : "#C73E1D",
    "MADLAD"      : "#3B1F2B",
}
sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams.update({
    "figure.dpi": 150, "figure.facecolor": "white",
    "axes.spines.top": False, "axes.spines.right": False,
    "font.family": "DejaVu Sans",
})
print("\u2713 Global theme applied")

In [ ]:
# ── Cell 1 · Load from Part B CSV + Tokenizers ────────────────────────────────
# Part B saved token_counts.csv
# Part C loads that file and loads tokenizers only (no full model weights)
# Data flow: A → B → C via saved CSVs. No kernel sharing between notebooks.

from transformers import AutoTokenizer

token_df = pd.read_csv("../part_b_token_analysis/token_counts.csv")
print(f"\u2713 token_df loaded from Part B: {len(token_df)} rows")

# Load tokenizers (lightweight — no GPU needed)
TOKENIZER_IDS = {
    "IndicTrans2" : ("ai4bharat/indictrans2-en-indic-1B", {"trust_remote_code": True}),
    "NLLB-200"    : ("facebook/nllb-200-distilled-600M", {}),
    "mT5"         : ("google/mt5-base", {}),
    "Helsinki"    : ("Helsinki-NLP/opus-mt-en-ta", {}),
    "MADLAD"      : ("google/madlad400-3b-mt", {}),
}

tokenizers = {}
for name, (model_id, kwargs) in TOKENIZER_IDS.items():
    tokenizers[name] = AutoTokenizer.from_pretrained(model_id, **kwargs)
    print(f"  \u2713 {name} tokenizer loaded \u2014 vocab size: {tokenizers[name].vocab_size:,}")

In [ ]:
# ── Cell 2 · Compute vocab_stats ──────────────────────────────────────────────
# Classifies each Tamil word as:
#   known      = single token (tokenizer "knows" this word)
#   fragmented = multiple tokens (word split into subwords)
#   unknown    = maps to UNK token (tokenizer has never seen this)
#
# Tamil is agglutinative — words have many suffixes attached.
# Poor tokenizers (like mT5 trained on English-heavy data) will
# fragment even common Tamil words into many subwords.

def compute_vocab_stats(tokenizer, sample_tamil_words):
    known = fragmented = unknown = 0
    unk_id = tokenizer.unk_token_id

    for word in sample_tamil_words:
        toks = tokenizer.encode(word, add_special_tokens=False)
        if not toks:
            continue
        if unk_id and toks[0] == unk_id:
            unknown += 1
        elif len(toks) == 1:
            known += 1
        else:
            fragmented += 1

    return {"known": known, "fragmented": fragmented, "unknown": unknown}


# Tamil test words: mix of simple, agglutinated, and domain-specific
# Agglutination examples: \u0bb5\u0ba8\u0bcd\u0ba4\u0bbf\u0bb0\u0bc1\u0b95\u0bcd\u0b95\u0bbf\u0bb1\u0bbe\u0ba9\u0bcd = "he has come" (one word, many morphemes)
sample_tamil_words = [
    # Simple common words
    "\u0bae\u0bb0\u0bae\u0bcd", "\u0ba8\u0ba9\u0bcd\u0bb1\u0bbf", "\u0b89\u0ba3\u0bb5\u0bc1", "\u0ba4\u0ba3\u0bcd\u0ba3\u0bc0\u0bb0\u0bcd", "\u0bae\u0b95\u0bcd\u0b95\u0bb3\u0bcd",
    # Medium complexity \u2014 verb forms
    "\u0baa\u0b9f\u0bbf\u0b95\u0bcd\u0b95\u0bbf\u0bb1\u0bbe\u0bb3\u0bcd", "\u0b9a\u0bc6\u0ba9\u0bcd\u0bb1\u0bbe\u0bb0\u0bcd\u0b95\u0bb3\u0bcd", "\u0baa\u0bc7\u0b9a\u0bc1\u0b95\u0bbf\u0bb1\u0bcb\u0bae\u0bcd",
    # High complexity \u2014 agglutinated
    "\u0bb5\u0ba8\u0bcd\u0ba4\u0bbf\u0bb0\u0bc1\u0b95\u0bcd\u0b95\u0bbf\u0bb1\u0bbe\u0ba9\u0bcd", "\u0b9a\u0bc6\u0baf\u0bcd\u0ba4\u0bc1\u0b95\u0bca\u0ba3\u0bcd\u0b9f\u0bbf\u0bb0\u0bc1\u0b95\u0bcd\u0b95\u0bbf\u0bb1\u0bbe\u0bb0\u0bcd\u0b95\u0bb3\u0bcd",
    # Domain specific
    "\u0b85\u0bb0\u0b9a\u0bbe\u0b99\u0bcd\u0b95\u0bae\u0bcd", "\u0b95\u0ba3\u0bbf\u0baa\u0bcd\u0baa\u0bca\u0bb1\u0bbf", "\u0b87\u0ba3\u0bc8\u0baf\u0bae\u0bcd", "\u0baa\u0bca\u0bb0\u0bc1\u0bb3\u0bbe\u0ba4\u0bbe\u0bb0\u0bae\u0bcd",
    # Places and proper nouns
    "\u0ba4\u0bae\u0bbf\u0bb4\u0bcd\u0ba8\u0bbe\u0b9f\u0bc1", "\u0b9a\u0bc6\u0ba9\u0bcd\u0ba9\u0bc8", "\u0b95\u0bcb\u0baf\u0bae\u0bcd\u0baa\u0bc1\u0ba4\u0bcd\u0ba4\u0bc2\u0bb0\u0bcd",
    # Rare / technical
    "\u0bae\u0b95\u0bcd\u0b95\u0bb3\u0bcd\u0ba4\u0bca\u0b95\u0bc8", "\u0bb5\u0bbf\u0bb5\u0b9a\u0bbe\u0baf\u0bbf", "\u0b9a\u0bc6\u0baf\u0bcd\u0ba4\u0bbf",
]

vocab_stats = {}
for name, tok in tokenizers.items():
    vocab_stats[name] = compute_vocab_stats(tok, sample_tamil_words)
    total = max(sum(vocab_stats[name].values()), 1)
    pct   = {k: round(v / total * 100, 1) for k, v in vocab_stats[name].items()}
    print(f"  {name:15s}: known={pct['known']}%  frag={pct['fragmented']}%  unk={pct['unknown']}%")

In [ ]:
# ── Cell 3 · Token Span Visualizer ────────────────────────────────────────────
# Shows how each model splits a Tamil word into subword tokens.
# Strips special tokens before rendering (\u2581 SentencePiece, ## WordPiece)

def clean_token_display(tok_str):
    tok_str = tok_str.replace("\u2581", "")
    tok_str = tok_str.replace("##", "")
    tok_str = tok_str.replace("<unk>", "?")
    tok_str = tok_str.strip()
    return tok_str if tok_str else "?"


def render_token_spans(word, tokens_per_model):
    color_pool = ["#2E86AB", "#A23B72", "#F18F01", "#C73E1D", "#3B1F2B",
                  "#44BBA4", "#E94F37", "#6A0572"]

    html = f"""
    <div style='font-family:monospace; margin:20px 0; padding:16px;
                border:1px solid #eee; border-radius:8px;'>
      <div style='font-size:20px; font-weight:bold; margin-bottom:14px;'>
        Word: <span style='color:#2E86AB'>{word}</span>
      </div>
    """
    for model_name, raw_tokens in tokens_per_model.items():
        tokens = [clean_token_display(t) for t in raw_tokens]
        tokens = [t for t in tokens if t]

        spans = ""
        for i, tok in enumerate(tokens):
            c = color_pool[i % len(color_pool)]
            spans += f"""
            <span style='background:{c}22; border:1.5px solid {c};
                         color:{c}; padding:3px 8px; border-radius:4px;
                         margin:2px; font-size:14px; font-weight:bold;
                         display:inline-block'>{tok}</span>"""

        count = len(tokens)
        label = f"({count} token{'s' if count != 1 else ''})"
        html += f"""
        <div style='margin:8px 0; display:flex; align-items:center; gap:12px;'>
          <span style='width:130px; font-weight:bold; font-size:13px;
                       color:#333;'>{model_name}</span>
          <span style='color:#999; font-size:12px; min-width:75px;'>{label}</span>
          <div style='flex:1;'>{spans}</div>
        </div>"""

    html += "</div>"
    return html


# Test on 3 complexity levels:
#   Simple  = one morpheme, common word
#   Medium  = 2-3 morphemes, frequent verb form
#   Complex = 4+ morphemes, heavily agglutinated verb
test_words = [
    ("Simple \u2014 \u0bae\u0bb0\u0bae\u0bcd (tree)",               "\u0bae\u0bb0\u0bae\u0bcd"),
    ("Medium \u2014 \u0baa\u0b9f\u0bbf\u0b95\u0bcd\u0b95\u0bbf\u0bb1\u0bbe\u0bb3\u0bcd (she studies)", "\u0baa\u0b9f\u0bbf\u0b95\u0bcd\u0b95\u0bbf\u0bb1\u0bbe\u0bb3\u0bcd"),
    ("Complex \u2014 \u0bb5\u0ba8\u0bcd\u0ba4\u0bbf\u0bb0\u0bc1\u0b95\u0bcd\u0b95\u0bbf\u0bb1\u0bbe\u0ba9\u0bcd (he has come)", "\u0bb5\u0ba8\u0bcd\u0ba4\u0bbf\u0bb0\u0bc1\u0b95\u0bcd\u0b95\u0bbf\u0bb1\u0bbe\u0ba9\u0bcd"),
]

for label, word in test_words:
    tokens_by_model = {
        name: tok.tokenize(word)
        for name, tok in tokenizers.items()
    }
    display(HTML(f"<h4>{label}</h4>"))
    display(HTML(render_token_spans(word, tokens_by_model)))

In [ ]:
# ── Cell 4 · VIZ C1 · Vocabulary Coverage Donut Charts ───────────────────────
# Donut charts: what % of Tamil words each model handles well
# known = best, fragmented = acceptable, unknown = worst

fig, axes = plt.subplots(1, 5, figsize=(20, 5))

for ax, (model_name, color) in zip(axes, MODEL_COLORS.items()):
    stats = vocab_stats[model_name]
    total = max(sum(stats.values()), 1)

    sizes  = [stats["known"] / total, stats["fragmented"] / total, stats["unknown"] / total]
    colors = [color, "#F18F01", "#C73E1D"]

    wedges, _, autotexts = ax.pie(
        sizes, colors=colors, autopct="%1.0f%%",
        startangle=90, pctdistance=0.75,
        wedgeprops=dict(width=0.5, edgecolor="white", linewidth=2),
    )
    for at in autotexts:
        at.set_fontsize(10)
        at.set_fontweight("bold")
    ax.set_title(model_name, fontsize=12, fontweight="bold", pad=10)

legend_elements = [
    mpatches.Patch(facecolor="#2E86AB", label="Known (1 token)"),
    mpatches.Patch(facecolor="#F18F01", label="Fragmented (split)"),
    mpatches.Patch(facecolor="#C73E1D", label="Unknown (UNK)"),
]
fig.legend(handles=legend_elements, loc="lower center", ncol=3, fontsize=12,
           bbox_to_anchor=(0.5, -0.05))
fig.suptitle("Tamil Vocabulary Coverage by Model", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.savefig("plots/partc_donut_coverage.png", bbox_inches="tight", dpi=150)
plt.show()

In [ ]:
# ── Cell 5 · VIZ C2 · Memory Footprint ───────────────────────────────────────
# Transformer attention memory scales quadratically with token count: O(n\u00b2)
# More tokens per sentence = more memory = harder to run on long texts

token_df["memory_score"] = token_df["target_token_count"] ** 2
mem_summary = token_df.groupby("model")["memory_score"].mean().sort_values()

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(
    mem_summary.index,
    mem_summary.values / 1000,
    color=[MODEL_COLORS[m] for m in mem_summary.index],
    edgecolor="white", linewidth=1.5, height=0.55,
)
for bar, val in zip(bars, mem_summary.values / 1000):
    ax.text(val + 0.3, bar.get_y() + bar.get_height() / 2,
            f"{val:.1f}k", va="center", fontsize=11, fontweight="bold")

ax.set_xlabel("Relative Attention Memory Score (token\u00b2 / 1000)", fontsize=12)
ax.set_title(
    "Transformer Memory Pressure by Model\n"
    "(More tokens = O(n\u00b2) attention cost \u2014 lower is more efficient)",
    fontsize=14, fontweight="bold",
)
ax.invert_yaxis()
plt.tight_layout()
plt.savefig("plots/partc_memory_footprint.png", bbox_inches="tight", dpi=150)
plt.show()

In [ ]:
# ── Cell 6 · VIZ C3 · Characters Per Token ────────────────────────────────────
# Higher avg characters per token = fewer splits = better Tamil tokenizer

chars_summary = token_df.groupby("model")["avg_word_length"].agg(["mean", "std"])

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(chars_summary))
bars = ax.bar(
    x, chars_summary["mean"],
    yerr=chars_summary["std"],
    color=list(MODEL_COLORS.values()),
    edgecolor="white", linewidth=1.5,
    capsize=5, width=0.55,
)
ax.set_xticks(x)
ax.set_xticklabels(chars_summary.index, fontsize=12)
ax.set_ylabel("Avg Characters per Token", fontsize=12)
ax.set_title(
    "Tamil Subword Quality: Characters per Token\n"
    "(Higher = less fragmentation = better Tamil tokenizer)",
    fontsize=14, fontweight="bold",
)

best_idx = chars_summary["mean"].argmax()
bars[best_idx].set_edgecolor("#2E86AB")
bars[best_idx].set_linewidth(3)
ax.text(
    best_idx,
    chars_summary["mean"].iloc[best_idx] + chars_summary["std"].iloc[best_idx] + 0.05,
    "\u2605 Best", ha="center", color="#2E86AB", fontweight="bold",
)
plt.tight_layout()
plt.savefig("plots/partc_chars_per_token.png", bbox_inches="tight", dpi=150)
plt.show()